In [1]:
!which python
!python --version

/home/agentolek/SSNE/.venv/bin/python
Python 3.13.12


In [2]:
import pandas as pd
import numpy as np

In [3]:
def rmsle(y_true,y_pred):
    n = len(y_true)
    msle = np.mean([(np.log(max(y_pred[i],0) + 1) - np.log(y_true[i] + 1)) ** 2.0 for i in range(n)])
    return np.sqrt(msle)

In [4]:
# check for NAs
data_df = pd.read_csv("data.csv")
print(f"Number of records: {len(data_df)}")
print(f"Number of rows with NAs: {data_df.isna().sum().sum()}")

Number of records: 10886
Number of rows with NAs: 0


In [5]:
# look at correlations to casual, registered, and cnt



In [6]:
from typing import Any

import torch
from torch import nn
# create simple neural network
class RegressionModel(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 32, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)

        self.layers = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)

In [7]:
# create Dataset class
class PandasBikeDataset(torch.utils.data.Dataset):
    def __init__(self, 
                 data_df: pd.DataFrame, 
                 pred_series: pd.Series, 
                 exclude_columns: list[str] | None = None) -> None:
        super().__init__()

        if exclude_columns is not None:
            self.data_df = data_df.drop(exclude_columns, axis=1)
        else:
            self.data_df = data_df
        
        self.labels = torch.Tensor(pred_series).to(dtype=torch.float)
    
    def __len__(self):
        return len(self.data_df)
    
    def __getitem__(self, index) -> Any:
        data = np.array(self.data_df.iloc[index].values)
        data = torch.from_numpy(data).to(dtype=torch.float)
        label = self.labels[index]
        return data, label

In [8]:
data_df.columns

Index(['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
       'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
       'casual', 'registered', 'cnt'],
      dtype='str')

In [ ]:
from sklearn.model_selection import train_test_split
# create Dataset
exclude_columns = ["instant", "casual", "registered", "cnt", "dteday"]
# exclude_columns = list(data_df.columns)
# exclude_columns.remove("casual")
# exclude_columns.remove("registered")
dataset = PandasBikeDataset(data_df=data_df,
                            pred_series=data_df["cnt"],
                            exclude_columns=exclude_columns)
                            
train_dataset, test_dataset = train_test_split(dataset, test_size=0.5, shuffle=True)

len(train_dataset)

5443

In [30]:
from tqdm import tqdm

model = RegressionModel(input_size=len(train_dataset[0][0]))
device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

# dataloaders
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

# hyperparameters
start_lr = 10e-3
epochs = 3

# optimizer, loss_fn
optim = torch.optim.AdamW(model.parameters(), lr=start_lr)
loss_fn = nn.MSELoss()

for epoch in tqdm(range(epochs)):
    running_loss = 0.0

    for i, data in enumerate(train_dataloader):

        X, y = data
        X, y = X.to(device), y.to(device).unsqueeze(dim=1)

        optim.zero_grad()

        preds = model(X)
        loss = loss_fn(preds, y)

        loss.backward()

        optim.step()
        running_loss += loss
        if (i+1) % 20 == 0:
            print('  batch {} loss: {}'.format(i + 1, running_loss / i))

100%|██████████| 3/3 [00:00<00:00, 19.06it/s]

  batch 20 loss: 12659.5546875
  batch 40 loss: 6646.3984375
  batch 60 loss: 4432.14697265625
  batch 80 loss: 3313.11572265625
  batch 20 loss: 3.1419217586517334
  batch 40 loss: 2.45833420753479
  batch 60 loss: 1.9558650255203247
  batch 80 loss: 1.649993658065796
  batch 20 loss: 0.6763191223144531
  batch 40 loss: 0.5726749897003174
  batch 60 loss: 0.5490090847015381
  batch 80 loss: 0.5278329253196716


In [25]:
# generate predictions on evaluation data using model
test_dataloader = torch.utils.data.DataLoader(test_dataset)

preds = torch.Tensor([])
targets = torch.Tensor([])

with torch.inference_mode():
    for X, y in tqdm(test_dataloader):
        X, y = X.to(device), y.to(device)

        curr_preds = model(X)
        preds = torch.cat((preds, curr_preds), dim=0)
        targets = torch.cat((targets, y), dim=0)


100%|██████████| 5443/5443 [00:00<00:00, 14137.37it/s]


In [26]:
pd.DataFrame(preds).to_csv("muchomorki.csv", index=False, header=None)

In [27]:
rmsle(targets.numpy(), preds.numpy())

np.float32(0.02720644)